In [1]:
import json
import os

final_nb = {
    "cells": [],
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.x"},
    },
    "nbformat": 4,
    "nbformat_minor": 5,
}

def md(text: str):
    final_nb["cells"].append({"cell_type": "markdown", "metadata": {}, "source": [text]})

def code(text: str):
    final_nb["cells"].append({"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": [text]})

def get_desktop_path(filename: str) -> str:
    """
    Returns a path to the user's Desktop.
    Handles both:
      - C:\\Users\\<you>\\Desktop
      - C:\\Users\\<you>\\OneDrive\\Desktop  (common on Windows)
    """
    home = os.path.expanduser("~")

    candidates = [
        os.path.join(home, "Desktop"),
        os.path.join(home, "OneDrive", "Desktop"),
        os.path.join(home, "OneDrive - Personal", "Desktop"),
    ]

    for d in candidates:
        if os.path.isdir(d):
            return os.path.join(d, filename)

    # Fallback: save in home folder if Desktop not found
    return os.path.join(home, filename)

# -------------------------
# Notebook content starts
# -------------------------

md(
"""# Simple RAG System (TF-IDF + FAISS + GPT-2) + Evaluation

**Goal:** Build a simple Retrieval-Augmented Generation (RAG) prototype that retrieves relevant text from a small knowledge base and uses **GPT-2** to generate an answer based on retrieved context.

This notebook follows the assignment steps:
1. Data preprocessing (dataset + cleaning + chunking)
2. Retrieval component (TF-IDF embeddings + FAISS index)
3. Query and retrieve top-k chunks
4. Generate responses with GPT-2
5. Evaluate the system (qualitative + required metrics)

**Knowledge base dataset:** SQuAD (Wikipedia paragraphs) from Hugging Face  
**Generator:** GPT-2
"""
)

code(
"""# Install dependencies (run once)
!pip -q install datasets faiss-cpu scikit-learn transformers torch
"""
)

code(
"""import re
import math
import numpy as np
import torch

import faiss
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

def normalize_for_retrieval(text: str) -> str:
    \"\"\"Lowercase + remove punctuation/special chars for TF-IDF retrieval.\"\"\"
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def chunk_into_sentence_blocks(text: str, sents_per_chunk: int = 3):
    \"\"\"Chunk text into ~2–3 sentence blocks (assignment-aligned chunking).\"\"\"
    text = re.sub(r"\\s+", " ", text).strip()
    sents = re.split(r"(?<=[.!?])\\s+", text)
    sents = [s.strip() for s in sents if s and s.strip()]
    chunks = []
    for i in range(0, len(sents), sents_per_chunk):
        chunk = " ".join(sents[i : i + sents_per_chunk]).strip()
        if len(chunk) >= 60:  # skip tiny chunks
            chunks.append(chunk)
    return chunks
"""
)

md("## Step 1: Data preprocessing (dataset selection, cleaning, chunking)")

code(
"""# Dataset: SQuAD (Wikipedia paragraphs)
# Dataset card: https://huggingface.co/datasets/rajpurkar/squad

ds = load_dataset("rajpurkar/squad", split="train")

# Keep it small for speed (adjust as needed)
N = 2000
subset = ds.select(range(N))

contexts = subset["context"]
questions = subset["question"]
answers = subset["answers"]

# Dedupe contexts to avoid repeated paragraphs
seen = set()
unique_contexts = []
for c in contexts:
    if c not in seen:
        unique_contexts.append(c)
        seen.add(c)

chunk_raw = []
chunk_clean = []

for ctx in unique_contexts:
    chunks = chunk_into_sentence_blocks(ctx, sents_per_chunk=3)
    for ch in chunks:
        chunk_raw.append(ch)
        chunk_clean.append(normalize_for_retrieval(ch))

print("Unique contexts:", len(unique_contexts))
print("Total chunks:", len(chunk_raw))
print("Sample chunk:\\n", chunk_raw[0][:350])
"""
)

md("## Step 2: Build the retrieval component (TF-IDF embeddings + FAISS index)")

code(
"""# TF-IDF embeddings (manageable size like the assignment example)
vectorizer = TfidfVectorizer(max_features=500)
X = vectorizer.fit_transform(chunk_clean)
dense_tfidf = X.toarray().astype(np.float32)

# FAISS index (IndexFlatL2 as in the assignment prompt)
dim = dense_tfidf.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(dense_tfidf)

print("TF-IDF shape:", dense_tfidf.shape)
print("FAISS index size:", index.ntotal)
"""
)

md("## Step 3: Query and retrieve top-k chunks")

code(
"""def retrieve(query: str, k: int = 3):
    q_vec = vectorizer.transform([normalize_for_retrieval(query)]).toarray().astype(np.float32)
    k = min(k, index.ntotal)
    distances, ids = index.search(q_vec, k)  # lower distance = better match

    results = []
    for dist, idx in zip(distances[0], ids[0]):
        results.append(
            {
                "chunk_id": int(idx),
                "distance": float(dist),
                "text": chunk_raw[int(idx)],
            }
        )
    return results

# Demo using an in-dataset question for reliable retrieval
demo_i = 0
user_query = questions[demo_i]
gold_answer = answers[demo_i]["text"][0] if answers[demo_i]["text"] else ""

retrieved_docs = retrieve(user_query, k=3)

print("Query:", user_query)
print("Gold answer (SQuAD):", gold_answer)
print("\\nTop-k retrieved chunks:")
for i, r in enumerate(retrieved_docs, 1):
    print(f"Top {i} | L2 distance={r['distance']:.4f}")
    print(r["text"][:260], "\\n")
"""
)

md("## Step 4: Generate responses using GPT-2")

code(
"""tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

def generate_answer(query: str, docs, max_new_tokens: int = 80) -> str:
    context = "\\n".join([f"- {d['text']}" for d in docs])
    prompt = (
        "Answer the question using the context.\\n\\n"
        f"Context:\\n{context}\\n\\n"
        f"Question: {query}\\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=900).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=False,              # deterministic for cleaner output
            max_new_tokens=max_new_tokens,
            no_repeat_ngram_size=2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return full_text.split("Answer:", 1)[-1].strip()

answer = generate_answer(user_query, retrieved_docs)
print("GENERATED ANSWER:\\n", answer)
"""
)

md("## Step 5A: Qualitative evaluation (required)")

code(
"""def run_qualitative_tests(idxs, k=3):
    for j, i in enumerate(idxs, 1):
        q = questions[i]
        gold = answers[i]["text"][0] if answers[i]["text"] else ""
        docs = retrieve(q, k=k)
        gen = generate_answer(q, docs, max_new_tokens=80)

        print("\\n" + "=" * 90)
        print(f"TEST {j} | idx={i}")
        print("Query:", q)
        print("Gold answer (SQuAD):", gold)

        print("\\nRetrieved chunks:")
        for t, d in enumerate(docs, 1):
            print(f"- Top {t} | dist={d['distance']:.4f} | {d['text'][:160]}...")

        print("\\nGenerated answer:")
        print(gen)

# Run a few tests (fast)
test_indices = [0, 5, 25, 50, 100]
run_qualitative_tests(test_indices, k=3)
"""
)

md(
"""### Evaluation Observations (Qualitative)

- Retrieval worked best when the query shared strong keywords with a chunk (for example, queries with distinctive named entities tend to retrieve on-topic chunks).
- Several queries failed because the top-k retrieved chunks did not contain the specific answer span (the system retrieved related context, but not the exact fact needed).
- When retrieval missed the answer, GPT-2 frequently hallucinated facts (made-up dates/names/places), even when prompted to use only the context.
- TF-IDF retrieval can struggle with abbreviations or rare tokens (e.g., short acronyms), which can lead to off-topic retrieval.
- Coherence was sometimes acceptable (readable sentences), but factual correctness depended heavily on whether the retrieved context contained the answer.

**Improvements (within assignment scope):**
- Increase knowledge base size (more contexts/chunks) and/or adjust chunk size for better retrieval precision.
- Increase `k` (e.g., 5) to raise the chance the correct answer span appears in retrieved text.
- Tune TF-IDF features (e.g., 1000–2000 features or character n-grams) to better handle abbreviations.
- Add a similarity/distance threshold: if retrieval confidence is low, return “I don’t know based on the knowledge base” instead of generating.
"""
)

md("## Step 5B: Required metrics (DDP example)")

code(
"""corpus = [
    "Distributed Data Parallel (DDP) allows multi-GPU training.",
    "DDP synchronizes gradients across GPUs.",
    "Using multiple GPUs increases training efficiency.",
    "Parallelization across GPUs is useful for large models.",
]

query = "What is Distributed Data Parallel (DDP) in PyTorch, and how is it useful in multi-GPU setups?"
response = (
    "Distributed Data Parallel (DDP) in PyTorch is a module that enables parallel training "
    "across multiple GPUs by distributing model replicas and splitting data across them."
)

# Relevancy: cosine similarity (TF-IDF between query and response)
vec = TfidfVectorizer(stop_words="english")
tfidf = vec.fit_transform([query, response])
relevancy = cosine_similarity(tfidf[0], tfidf[1])[0, 0]

# Completeness: proportion of keywords from query found in response
q_tokens = set(vec.build_analyzer()(query))
r_tokens = set(vec.build_analyzer()(response))
keywords = {t for t in q_tokens if len(t) > 2}
completeness = (len(keywords & r_tokens) / len(keywords)) if keywords else 0.0

# Coherence: perplexity score (GPT-2)
with torch.no_grad():
    inputs = tokenizer(response, return_tensors="pt")
    out = model(**inputs.to(DEVICE), labels=inputs["input_ids"].to(DEVICE))
    loss = out.loss.item()
perplexity = math.exp(loss)

print(f"Relevancy (cosine similarity query↔response): {relevancy:.4f}")
print(f"Completeness (keyword coverage): {completeness:.4f}")
print(f"Coherence (perplexity, lower is better): {perplexity:.2f}")

print("\\nKeywords from query (sample):", sorted(list(keywords))[:20])
print("Matched keywords:", sorted(list((keywords & r_tokens)))[:20])
"""
)

md("## Optional experiment: chunk size impact on retrieval")

code(
"""def build_chunks_with_sents_per_chunk(sents_per_chunk: int):
    raw = []
    clean = []
    for ctx in unique_contexts:
        chunks = chunk_into_sentence_blocks(ctx, sents_per_chunk=sents_per_chunk)
        for ch in chunks:
            raw.append(ch)
            clean.append(normalize_for_retrieval(ch))
    return raw, clean

def retrieval_answer_hit_rate(sents_per_chunk: int, sample_q: int = 200, k: int = 3):
    raw, clean = build_chunks_with_sents_per_chunk(sents_per_chunk)

    vec_local = TfidfVectorizer(max_features=500)
    X_local = vec_local.fit_transform(clean)
    dense_local = X_local.toarray().astype(np.float32)

    idx_local = faiss.IndexFlatL2(dense_local.shape[1])
    idx_local.add(dense_local)

    hits, total = 0, 0

    for i in range(min(sample_q, len(questions))):
        q = questions[i]
        golds = answers[i]["text"]
        if not golds:
            continue
        gold = golds[0].strip().lower()
        if not gold:
            continue

        q_vec = vec_local.transform([normalize_for_retrieval(q)]).toarray().astype(np.float32)
        distances, ids = idx_local.search(q_vec, min(k, idx_local.ntotal))

        retrieved_text = " ".join([raw[int(j)].lower() for j in ids[0]])
        if gold in retrieved_text:
            hits += 1
        total += 1

    return hits / total if total else 0.0

for spc in [2, 3, 4]:
    rate = retrieval_answer_hit_rate(spc, sample_q=200, k=3)
    print(f"sents_per_chunk={spc} | answer-hit-rate@3 over 200 Qs: {rate:.3f}")
"""
)

md(
"""## References and links (APA-style)

- Hugging Face. (n.d.). *SQuAD dataset card (rajpurkar/squad).* https://huggingface.co/datasets/rajpurkar/squad  
- Hugging Face. (n.d.). *GPT-2 model card.* https://huggingface.co/gpt2  
- Hugging Face. (n.d.). *Datasets documentation.* https://huggingface.co/docs/datasets/  
- Hugging Face. (n.d.). *Transformers documentation.* https://huggingface.co/docs/transformers/  
- Adrian Twarog. (2023, June 30). *OpenAI Embeddings and Vector Databases Crash Course* [Video]. YouTube. https://www.youtube.com/watch?v=ySus5ZS0b94  
- Mixture. (2019, April 6). *Generate Text using OpenAIGPT2 in Python* [Video]. YouTube. https://www.youtube.com/watch?v=XDzByaM39AE  
- sam_mk87. (2024, January 27). *Mastering RAG Evaluation: Metrics and Methods | Retrieval-Augmented Generation* [Video]. YouTube. https://www.youtube.com/watch?v=s-7r86OLdd4  
"""
)

# -------------------------
# Save to Desktop
# -------------------------
filename = "Week7_RAG_FINAL.ipynb"
out_path = get_desktop_path(filename)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final_nb, f, ensure_ascii=False, indent=1)

print("Saved notebook to:", out_path)

Saved notebook to: C:\Users\shweiss\Desktop\Week7_RAG_FINAL.ipynb
